In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/krupalpatel07/dell-daily-data/DELL.csv


In [2]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'iframe'

from IPython.display import HTML, display

display(HTML("""
<div style='background:linear-gradient(90deg,#0f172a,#1e293b);
padding:25px;border-radius:15px'>
<h1 style='color:white'>🚀 DELL ALPHA LAB</h1>
<h3 style='color:#93c5fd'>Fractal Markets • Regime Detection • Quant Strategy</h3>
</div>
"""))

In [3]:
# ============================================
# LOAD DATA
# ============================================

df = pd.read_csv("/kaggle/input/datasets/krupalpatel07/dell-daily-data/DELL.csv")

df.columns = [c.strip().title() for c in df.columns]

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')


In [4]:
# ============================================
# RETURNS
# ============================================

df['Return'] = df['Close'].pct_change()

In [6]:
# ============================================
# HURST EXPONENT
# ============================================

def hurst(ts):
    lags = range(2,20)
    tau = [np.std(np.subtract(ts[lag:], ts[:-lag])) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau),1)
    return poly[0]*2

hurst_value = hurst(df['Close'].values)

# ============================================
# REGIMES
# ============================================

df['MA50'] = df['Close'].rolling(50).mean()

df['Regime'] = np.where(
    df['Close'] > df['MA50'],
    'Bull',
    'Bear'
)


In [7]:
# ============================================
# STRATEGY
# ============================================

df['Signal'] = np.where(
    df['Close'] > df['MA50'],
    1,
    0
)

df['Strategy_Return'] = (
    df['Signal'].shift(1)
    * df['Return']
)


In [8]:

# ============================================
# PERFORMANCE
# ============================================

strategy = df['Strategy_Return'].fillna(0)

cagr = (
    (1+strategy).cumprod().iloc[-1]
) ** (252/len(df)) - 1

sharpe = (
    strategy.mean()
    / strategy.std()
) * np.sqrt(252)

cum = (1+strategy).cumprod()

roll_max = cum.cummax()

drawdown = (cum-roll_max)/roll_max

max_dd = drawdown.min()

print("="*60)
print("DELL QUANT REPORT")
print("="*60)
print(f"Hurst Exponent : {hurst_value:.3f}")
print(f"CAGR           : {cagr:.2%}")
print(f"Sharpe Ratio   : {sharpe:.2f}")
print(f"Max Drawdown   : {max_dd:.2%}")


DELL QUANT REPORT
Hurst Exponent : 0.926
CAGR           : 38.75%
Sharpe Ratio   : 1.11
Max Drawdown   : -39.86%


In [9]:
# ============================================
# DASHBOARD
# ============================================

fig = make_subplots(
    rows=3,
    cols=1,
    vertical_spacing=0.06,
    subplot_titles=(
        "Dell Price & Regimes",
        "Daily Returns",
        "Strategy Equity Curve"
    )
)

fig.add_trace(
    go.Scatter(
        x=df['Date'],
        y=df['Close'],
        name='Close'
    ),
    row=1,col=1
)

fig.add_trace(
    go.Scatter(
        x=df['Date'],
        y=df['MA50'],
        name='MA50'
    ),
    row=1,col=1
)

fig.add_trace(
    go.Bar(
        x=df['Date'],
        y=df['Return'],
        name='Returns'
    ),
    row=2,col=1
)

fig.add_trace(
    go.Scatter(
        x=df['Date'],
        y=(1+strategy).cumprod(),
        name='Equity Curve'
    ),
    row=3,col=1
)

fig.update_layout(
    template='plotly_dark',
    height=1000,
    title=f"""
    DELL QUANTUM RESEARCH DASHBOARD
    <br>
    Hurst={hurst_value:.2f}
    | Sharpe={sharpe:.2f}
    | CAGR={cagr:.2%}
    """
)

fig.show()

# ============================================
# EXECUTIVE SUMMARY
# ============================================

print("\nEXECUTIVE SUMMARY")
print("-"*60)

if hurst_value > 0.5:
    print("Market exhibits trending behaviour.")
else:
    print("Market exhibits mean-reverting behaviour.")

if sharpe > 1:
    print("Strategy demonstrates institutional-quality risk-adjusted returns.")
else:
    print("Strategy requires further alpha enhancement.")

print("Research completed successfully.")


EXECUTIVE SUMMARY
------------------------------------------------------------
Market exhibits trending behaviour.
Strategy demonstrates institutional-quality risk-adjusted returns.
Research completed successfully.
